# Forecast uncertainty exploration

Exploration only; this notebook is not imported by application code. It compares forecasting baselines on a chronological held-out segment and inspects DS-DGP intervals when a trained artifact is present.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from ml.forecasting.calibration import coverage, ece, nll

cwd = Path.cwd()
parents = (cwd, *cwd.parents)
repo_root = next(
    (parent for parent in parents if (parent / 'data/processed').exists()),
    cwd,
)
path = repo_root / 'data/processed/sih_product/splits/forecast_train.csv'
frame = pd.read_csv(path).dropna(subset=['capacity_mw']).reset_index(drop=True).iloc[:600]
values = pd.to_numeric(frame.capacity_mw, errors='raise').to_numpy(dtype=float)
split = int(len(values) * 0.8)
history, heldout = values[:split], values[split:]
print('training rows:', len(history), 'held-out rows:', len(heldout))

In [ ]:
from ml.forecasting.baselines import arima, lstm, sparse_gp

observed_history = np.concatenate((history, heldout))
test_features = pd.DataFrame({
    'target_lag_1': observed_history[split - 1:-1],
    'target_lag_2': observed_history[split - 2:-2],
    'forecast_horizon_step': np.ones(len(heldout), dtype=float),
})
train_frame = pd.DataFrame({'target': history})
results = {
    'ARIMA': arima.fit_predict(train_frame, test_features, seed=42),
    'LSTM': lstm.fit_predict(train_frame, test_features, seed=42),
    'Sparse GP': sparse_gp.fit_predict(train_frame, test_features, seed=42),
}
summary = []
for name, result in results.items():
    std = np.maximum((result.upper - result.lower) / (2 * 1.959963984540054), 1e-8)
    summary.append(
        {
            'model': name,
            'coverage_95': coverage(heldout, result.lower, result.upper),
            'ece': ece(heldout, result.mean, std),
            'nll': nll(heldout, result.mean, std),
        }
    )
display(pd.DataFrame(summary))

In [ ]:
artifact = repo_root / 'ml/artifacts/ds_dgp'
if (artifact / 'model.pt').is_file():
    from ml.forecasting.ds_dgp.model import load_model, predict
    model = load_model(artifact)
    future_index = np.arange(
        model.training_rows, model.training_rows + 24, dtype=float
    )
    mean, lower, upper = predict(model, future_index, seed=42)
    print('24-step mean range:', float(mean.min()), float(mean.max()))
    print('all intervals ordered:', bool(np.all(lower <= mean) and np.all(mean <= upper)))
else:
    print(
        'Train DS-DGP with ml/forecasting/configs/ds_dgp.yaml to inspect learned intervals.'
    )